In [1]:
import pandas as pd

In [6]:
nonlinear_no_conf_f50_s1000_p50=pd.read_parquet("data/synthetic/nonlinear_no_conf_f50_s1000_p50.parquet")

In [7]:
nonlinear_no_conf_f50_s1000_p50.describe()

,X0,X1,X2,X3,X4,X5,X6,X7,X8,X9,...,X41,X42,X43,X44,X45,X46,X47,X48,X49,Y
count,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,...,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000
mean,-0.000310,-0.009389,-0.001355,-0.006198,-0.032337,-0.023194,0.006788,0.008811,-0.007414,-0.013403,...,0.011554,0.003309,0.002734,0.009450,-0.030436,-0.026886,0.025564,-0.008969,0.019389,0.000766
std,0.492045,1.119219,1.122986,1.136105,1.137383,1.139523,1.121283,1.124999,1.117602,1.144462,...,1.124767,1.110198,1.123723,1.106643,1.095052,1.101681,1.113908,1.118443,1.134721,1.113413
min,-1.444136,-5.435189,-2.931854,-2.339673,-7.437984,-2.178001,-2.623713,-4.918708,-3.061491,-6.501972,...,-6.921434,-9.661765,-9.365404,-9.243679,-7.484635,-8.407257,-7.479539,-7.072082,-7.832814,-10.024985
25%,-0.343967,-0.448908,-0.828294,-0.775572,-0.615658,-0.524035,-0.903636,-0.652930,-0.763660,-0.463805,...,-0.449801,-0.319568,-0.236326,-0.383420,-0.283582,-0.387900,-0.351000,-0.417542,-0.366507,-0.313232
50%,0.006245,0.239350,0.124317,-0.285685,0.001996,-0.139307,-0.104009,0.055479,0.078307,0.108104,...,-0.120065,0.053523,0.106963,-0.019263,0.069292,-0.046388,0.013468,-0.098763,0.016342,0.072048
75%,0.321257,0.733249,0.866357,0.541939,0.689709,0.256918,0.866962,0.803305,0.828473,0.532370,...,0.267394,0.434996,0.447471,0.329143,0.412749,0.309837,0.358571,0.301126,0.352281,0.394474
max,1.482505,2.237195,2.647376,5.897578,2.850514,10.872743,3.539966,3.039964,2.931510,6.470101,...,8.576768,8.635379,4.360501,8.326954,7.531972,7.445241,8.196714,10.060670,7.827716,9.597541


In [ ]:
nonlinear_no_conf_f50_s1000_p50

In [9]:
from predictive_models.predictive_models import LGBMRegressor


model_path = "models/linear_conf_f10_s1000_p50_lgbm"
model = LGBMRegressor.load(str(model_path))

LightGBM model loaded from models/linear_conf_f10_s1000_p50_lgbm.pkl


In [13]:
model.selected_features

['X0', 'X1', 'X2', 'X3', 'X4', 'X5', 'X6', 'X7', 'X8', 'X9']

In [11]:
model.model.feature_name_

['X0', 'X1', 'X2', 'X3', 'X4', 'X5', 'X6', 'X7', 'X8', 'X9']

In [18]:
type(model)

predictive_models.predictive_models.LGBMRegressor

In [17]:
type(model.model)

lightgbm.sklearn.LGBMRegressor

# Shapflow Package Example

Testing the shapflow package implementation with nonlinear_no_conf_f50_s1000_p50 dataset.

**Steps:**
1. Load the dataset and trained model
2. Load the causal graph (adjacency matrix)
3. Use shapflow to calculate SHAP values with causal structure
4. Compare with our custom implementation

In [1]:
# Correct imports from shapflow.flow
from shapflow.flow import (
    Node, CreditFlow, Graph, GraphExplainer,
    get_source_nodes, topo_sort, flatten_graph, eval_graph,
    boundary_graph, single_source_graph, viz_graph, save_graph,
    ParallelCreditFlow, translator,
    group_nodes, build_feature_graph,
    CausalLinks, create_xgboost_f,
    edge_credits2edge_credit, node_dict2str_dict
)
import numpy as np
import pandas as pd
from pathlib import Path

print("✅ Shapflow imports successful!")

✅ Shapflow imports successful!


## 1. Load Dataset and Model

In [2]:
# Load dataset
data = pd.read_parquet("data/synthetic/nonlinear_no_conf_f50_s1000_p50.parquet")
print(f"Dataset shape: {data.shape}")
print(f"Features: {[col for col in data.columns if col != 'Y']}")

# Load processed train/test splits
train_data = pd.read_parquet("data/processed/nonlinear_no_conf_f50_s1000_p50_train.parquet")
test_data = pd.read_parquet("data/processed/nonlinear_no_conf_f50_s1000_p50_test.parquet")

X_train = train_data.drop('Y', axis=1)
y_train = train_data['Y']
X_test = test_data.drop('Y', axis=1)
y_test = test_data['Y']

print(f"Train shape: {X_train.shape}, Test shape: {X_test.shape}")

Dataset shape: (1000, 51)
Features: ['X0', 'X1', 'X2', 'X3', 'X4', 'X5', 'X6', 'X7', 'X8', 'X9', 'X10', 'X11', 'X12', 'X13', 'X14', 'X15', 'X16', 'X17', 'X18', 'X19', 'X20', 'X21', 'X22', 'X23', 'X24', 'X25', 'X26', 'X27', 'X28', 'X29', 'X30', 'X31', 'X32', 'X33', 'X34', 'X35', 'X36', 'X37', 'X38', 'X39', 'X40', 'X41', 'X42', 'X43', 'X44', 'X45', 'X46', 'X47', 'X48', 'X49']
Train shape: (800, 50), Test shape: (200, 50)


In [3]:
# Load trained model
from predictive_models.predictive_models import LGBMRegressor

model_path = "models/nonlinear_no_conf_f50_s1000_p50_lgbm"
model = LGBMRegressor.load(str(model_path))

print(f"Model type: {type(model)}")
print(f"Selected features: {model.selected_features}")
print(f"Number of selected features: {len(model.selected_features)}")

LightGBM model loaded from models/nonlinear_no_conf_f50_s1000_p50_lgbm.pkl
Model type: <class 'predictive_models.predictive_models.LGBMRegressor'>
Selected features: ['X0', 'X1', 'X2', 'X4', 'X5', 'X9', 'X10', 'X11', 'X13', 'X14', 'X16', 'X17', 'X18', 'X19', 'X22', 'X23', 'X24', 'X25', 'X27', 'X28', 'X29', 'X30', 'X31', 'X32', 'X33', 'X34', 'X35', 'X36', 'X37', 'X39', 'X40', 'X42', 'X43', 'X46', 'X49']
Number of selected features: 35


## 2. Load Causal Graph

In [4]:
# Load the PC adjacency matrix (train data)
adj_matrix_path = "data/causal/nonlinear_no_conf_f50_s1000_p50_pc_train_adjacency.npy"
adjacency_matrix = np.load(adj_matrix_path)

print(f"Adjacency matrix shape: {adjacency_matrix.shape}")
print(f"Number of edges: {np.sum(adjacency_matrix != 0)}")
print(f"Graph density: {np.sum(adjacency_matrix != 0) / (adjacency_matrix.shape[0]**2):.3f}")

Adjacency matrix shape: (50, 50)
Number of edges: 218
Graph density: 0.087


## 3. Build Shapflow Graph

Convert adjacency matrix to shapflow Graph format.

In [5]:
# Build feature graph from adjacency matrix
# adjacency_matrix[i,j] = 1 means there's an edge from feature i to feature j

feature_names = [f'X{i}' for i in range(adjacency_matrix.shape[0])]

# Step 1: Create all feature nodes first (without parent relationships)
nodes = {}
for i, name in enumerate(feature_names):
    # Create node with just name - we'll add parents in step 2
    nodes[name] = Node(name=name)

# Step 2: Add parent-child relationships between features
for i, name in enumerate(feature_names):
    # Find parents of this node (incoming edges)
    parent_indices = [j for j in range(adjacency_matrix.shape[0]) 
                     if adjacency_matrix[j, i] != 0]
    
    # Add each parent node to this node's args
    for parent_idx in parent_indices:
        parent_name = feature_names[parent_idx]
        nodes[name].add_arg(nodes[parent_name])

# Step 3: Create target node (model prediction)
# The target node needs a function that takes feature values and returns prediction
def target_function(**kwargs):
    """Model prediction function for the target node"""
    return model_predict(kwargs)

# Create target node - it depends on all features
target_node = Node(
    name='Y',
    f=target_function,
    is_target_node=True
)

# Add all feature nodes as parents of the target
for feature_name in feature_names:
    target_node.add_arg(nodes[feature_name])

# Add target to nodes dict
nodes['Y'] = target_node

# Create graph
graph = Graph(nodes=list(nodes.values()))

print(f"Graph created with {len(nodes)} nodes ({len(feature_names)} features + 1 target)")
print(f"Number of edges in graph: {sum(len(node.args) for node in graph.nodes)}")
print(f"Source nodes (no parents): {get_source_nodes(graph)}")
print(f"Target nodes: {[n.name for n in graph.nodes if n.is_target_node]}")

Graph created with 51 nodes (50 features + 1 target)
Number of edges in graph: 268
Source nodes (no parents): [X0]
Target nodes: ['Y']


## 4. Prepare Model Function for Shapflow

Shapflow needs a function that takes feature values and returns predictions.

In [6]:
# Create a prediction function compatible with shapflow
def model_predict(X_dict):
    """
    Shapflow passes features as a dictionary {feature_name: value}
    We need to convert to DataFrame and use our model wrapper.
    """
    # Convert dict to DataFrame with correct column order
    X_df = pd.DataFrame([X_dict])
    
    # Use the model wrapper's predict method (handles feature selection)
    prediction = model.predict(X_df)
    
    return prediction[0]

# Test the function
test_instance = X_test.iloc[0].to_dict()
test_pred = model_predict(test_instance)
print(f"Test prediction: {test_pred:.4f}")
print(f"Actual y_test[0]: {y_test.iloc[0]:.4f}")

Test prediction: -0.0018
Actual y_test[0]: 0.5195


## 5. Calculate Shapley Values with Shapflow

Use GraphExplainer to compute causal Shapley values.

In [7]:
# First, let's check what parameters GraphExplainer actually expects
import inspect
sig = inspect.signature(GraphExplainer.__init__)
print("GraphExplainer.__init__ signature:")
print(sig)
print("\nParameters:")
for param_name, param in sig.parameters.items():
    if param_name != 'self':
        default_val = param.default if param.default != inspect.Parameter.empty else 'REQUIRED'
        print(f"  {param_name}: {default_val}")

GraphExplainer.__init__ signature:
(self, graph, bg, nruns=100, silent=False)

Parameters:
  graph: REQUIRED
  bg: REQUIRED
  nruns: 100
  silent: False


In [12]:
# Select a test instance
instance_idx = 0
x_explain = X_test.iloc[instance_idx].to_dict()

print(f"Explaining instance {instance_idx}")
print(f"True Y: {y_test.iloc[instance_idx]:.4f}")
print(f"Predicted Y: {model_predict(x_explain):.4f}")

# Create GraphExplainer with correct parameters
# bg = background data as DataFrame (NOT list of dicts)
# The column names must match the node names in the graph (including 'Y')
bg_data = X_train.head(100).copy()

# Add Y column to background data (compute predictions for all bg instances)
print("\nComputing predictions for background data...")
bg_data['Y'] = [model_predict(row.to_dict()) for _, row in bg_data.iterrows()]

print(f"Background data shape: {bg_data.shape}")
print(f"Background data columns: {list(bg_data.columns)[:5]}... Y")
print(f"Graph nodes: {[n.name for n in graph.nodes[:3]]}... Y")

explainer = GraphExplainer(
    graph=graph,
    bg=bg_data,      # DataFrame with columns matching ALL node names (features + target)
    nruns=100,       # Number of Monte Carlo samples
    silent=False     # Show progress
)

print("✅ GraphExplainer created")
print(f"Type: {type(explainer)}")
print(f"Available methods: {[m for m in dir(explainer) if not m.startswith('_')]}")

Explaining instance 0
True Y: 0.5195
Predicted Y: -0.0018

Computing predictions for background data...
Background data shape: (100, 51)
Background data columns: ['X0', 'X1', 'X2', 'X3', 'X4']... Y
Graph nodes: ['X38', 'Y', 'X19']... Y
✅ GraphExplainer created
Type: <class 'shapflow.flow.GraphExplainer'>
Available methods: ['bg', 'graph', 'nruns', 'prepare_graph', 'set_noise_sampler', 'shap_values', 'silent']


In [15]:
graph.nodes

[X38,
 Y,
 X19,
 X32,
 X5,
 X13,
 X26,
 X45,
 X39,
 X20,
 X6,
 X33,
 X14,
 X27,
 X46,
 X40,
 X0,
 X7,
 X21,
 X34,
 X15,
 X28,
 X47,
 X8,
 X41,
 X22,
 X35,
 X16,
 X9,
 X29,
 X48,
 X42,
 X23,
 X4,
 X36,
 X17,
 X10,
 X2,
 X30,
 X49,
 X25,
 X43,
 X1,
 X24,
 X11,
 X37,
 X3,
 X18,
 X31,
 X44,
 X12]

In [16]:
# Calculate Shapley values
# Note: shap_values() expects a DataFrame, not a dict

try:
    # Check available methods
    methods = [m for m in dir(explainer) if not m.startswith('_') and callable(getattr(explainer, m))]
    print(f"Available methods: {methods}\n")
    
    # Convert test instance to DataFrame (shap_values expects DataFrame)
    x_explain_df = X_test.iloc[[instance_idx]].copy()
    
    # Add target column 'Y' with the predicted value (required by shapflow)
    x_explain_df['Y'] = model_predict(X_test.iloc[instance_idx].to_dict())
    
    print(f"Explaining data shape: {x_explain_df.shape}")
    print(f"Columns: {list(x_explain_df.columns)[:5]}... Y")
    print(f"Y value: {x_explain_df['Y'].values[0]:.4f}")
    
    # Debug: Check target node before calling shap_values
    print("\n🔍 Debug: Checking graph state...")
    print(f"Graph nodes count: {len(explainer.graph.nodes)}")
    target_nodes = [n for n in explainer.graph.nodes if n.is_target_node]
    print(f"Target nodes found: {[n.name for n in target_nodes]}")
    print(f"Target node count: {len(target_nodes)}")
    
    if len(target_nodes) == 0:
        print("\n⚠️  WARNING: No target node found in explainer.graph!")
        print("Trying to check if original graph still has target...")
        orig_target = [n for n in graph.nodes if n.is_target_node]
        print(f"Original graph target nodes: {[n.name for n in orig_target]}")
    
    # Calculate shapley values
    shapley_values = explainer.shap_values(x_explain_df)
    
    print("\n📊 Shapley Values:")
    print("="*60)
    
    # shapley_values might be a DataFrame or dict - check type
    print(f"Result type: {type(shapley_values)}")
    
    if isinstance(shapley_values, pd.DataFrame):
        # Drop Y if it's in there
        shapley_values_clean = shapley_values.drop('Y', errors='ignore')
        shapley_df = shapley_values_clean.T  # Transpose if needed
        shapley_df.columns = ['Shapley Value']
        shapley_df['Feature'] = shapley_df.index
        shapley_df = shapley_df[['Feature', 'Shapley Value']]
    elif isinstance(shapley_values, dict):
        # Remove Y from dict if present
        shapley_values_clean = {k: v for k, v in shapley_values.items() if k != 'Y'}
        shapley_df = pd.DataFrame({
            'Feature': list(shapley_values_clean.keys()),
            'Shapley Value': list(shapley_values_clean.values())
        })
    else:
        # It might be a numpy array
        shapley_df = pd.DataFrame({
            'Feature': X_test.columns,
            'Shapley Value': shapley_values.flatten() if hasattr(shapley_values, 'flatten') else shapley_values
        })
    
    shapley_df = shapley_df.sort_values('Shapley Value', ascending=False, key=abs)
    print(shapley_df.head(10))
    
except Exception as e:
    print(f"❌ Error: {e}")
    print(f"Error type: {type(e).__name__}")
    import traceback
    traceback.print_exc()

Available methods: ['prepare_graph', 'set_noise_sampler', 'shap_values']

Explaining data shape: (1, 51)
Columns: ['X0', 'X1', 'X2', 'X3', 'X4']... Y
Y value: -0.0018

🔍 Debug: Checking graph state...
Graph nodes count: 51
Target nodes found: ['Y']
Target node count: 1


bruteforce sampling:   0%|          | 0/100 [00:00<?, ?it/s]

❌ Error: 0 target node, need 1
Error type: AssertionError



Traceback (most recent call last):
  File "/var/folders/qp/vvw_x_0134j4ssq5wp352csc0000gn/T/ipykernel_11026/2220194418.py", line 33, in <module>
    shapley_values = explainer.shap_values(x_explain_df)
                     ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/juanrios/Documents/master_thesis/.venv/lib/python3.11/site-packages/shapflow/flow.py", line 1233, in shap_values
    cf.run(method, len_bg=len(self.bg))
  File "/Users/juanrios/Documents/master_thesis/.venv/lib/python3.11/site-packages/shapflow/flow.py", line 784, in run
    self.run_bruteforce_sampling()
  File "/Users/juanrios/Documents/master_thesis/.venv/lib/python3.11/site-packages/shapflow/flow.py", line 730, in run_bruteforce_sampling
    self.reset()
  File "/Users/juanrios/Documents/master_thesis/.venv/lib/python3.11/site-packages/shapflow/flow.py", line 509, in reset
    self.graph.reset()
  File "/Users/juanrios/Documents/master_thesis/.venv/lib/python3.11/site-packages/shapflow/flow.py", line 263, in res

## 6. Compare with Custom Implementation

Load results from our custom ShapleyFlow implementation.

In [10]:
# Load custom implementation results
import json

custom_results_path = "data/explainability/nonlinear_no_conf_f50_s1000_p50/lgbm/pc/flow_shapley_values.json"

try:
    with open(custom_results_path, 'r') as f:
        custom_results = json.load(f)
    
    # Extract Shapley values for the same instance
    custom_shapley = custom_results['shapley_values'][instance_idx]
    
    print("\n📊 Custom Implementation Shapley Values:")
    print("="*60)
    custom_df = pd.DataFrame({
        'Feature': list(custom_shapley.keys()),
        'Shapley Value': list(custom_shapley.values())
    })
    custom_df = custom_df.sort_values('Shapley Value', ascending=False, key=abs)
    print(custom_df.head(10))
    
except FileNotFoundError:
    print("❌ Custom implementation results not found")
except Exception as e:
    print(f"❌ Error loading custom results: {e}")

❌ Custom implementation results not found


## 7. Visualization

Compare the two implementations side by side.

In [11]:
import matplotlib.pyplot as plt
import seaborn as sns

# Create comparison plot if both results available
try:
    # Merge dataframes
    comparison = pd.merge(
        shapley_df, custom_df, 
        on='Feature', 
        suffixes=('_shapflow', '_custom')
    )
    
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    # Shapflow results
    top_10_shapflow = shapley_df.head(10).sort_values('Shapley Value')
    axes[0].barh(top_10_shapflow['Feature'], top_10_shapflow['Shapley Value'])
    axes[0].set_title('Shaflow Package - Top 10 Features', fontsize=14, fontweight='bold')
    axes[0].set_xlabel('Shapley Value')
    axes[0].grid(True, alpha=0.3)
    
    # Custom implementation results
    top_10_custom = custom_df.head(10).sort_values('Shapley Value')
    axes[1].barh(top_10_custom['Feature'], top_10_custom['Shapley Value'])
    axes[1].set_title('Custom Implementation - Top 10 Features', fontsize=14, fontweight='bold')
    axes[1].set_xlabel('Shapley Value')
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Correlation analysis
    corr = comparison[['Shapley Value_shapflow', 'Shapley Value_custom']].corr()
    print(f"\n📈 Correlation between implementations: {corr.iloc[0,1]:.4f}")
    
except Exception as e:
    print(f"Could not create comparison plot: {e}")

Could not create comparison plot: name 'shapley_df' is not defined
